# Flux relight example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from scenes import *
from losses.flux_relight_loss import FLUXKontextRelighter, FluxLoss
from utils.image.display import display_tensor
from losses.flux_relight_loss import FluxLoss, RelightImageCache
from losses.image_image import SSIMLoss, MSELossWithReferenceImage, LPIPSLoss

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.06
n_iter = 300

In [ ]:
global_seed = 1 # Can be None

In [ ]:
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
relighter = FLUXKontextRelighter(seed=global_seed)

In [ ]:
scene = SciFiRobotScene(include_alpha_mask=True)
display_tensor(color_space_converter(scene.get_combined_image(None).permute(2, 1, 0)))
images_cache = RelightImageCache(relighter, image_to_relight=scene.get_combined_image(color_space_converter).permute(2, 0, 1).unsqueeze(0).to(device))


In [ ]:
target_text = "aesthetic, golden hour lighting"
num_results = 4
# criterion_lpips = FluxLoss(cache=images_cache, image_comparison_criterion_cls=LPIPSLoss, target_text=target_text, num_relighted_images=num_results, display=True)
criterion_mse = FluxLoss(cache=images_cache, image_comparison_criterion_cls=MSELossWithReferenceImage, target_text=target_text, num_relighted_images=num_results, display=True)
# criterion_ssim = FluxLoss(cache=images_cache, image_comparison_criterion_cls=SSIMLoss, target_text=target_text, num_relighted_images=3, display=True)

In [ ]:
import importlib
import os
importlib.reload(importlib.import_module("utils.train"))
from utils.train import train_with_criterion

for criterion, title in [
    (criterion_mse, "MSE"),
    # (criterion_ssim, "SSIM"),
    # (criterion_lpips, "LPIPS"),
]:
    print(f"Starting training with criterion: {title}")
    train_with_criterion(
        scene,
        lr, n_iter, criterion,
        patience=40,
        starting_multiplier_std=(0.1, 0.1, 0.1),
        output_subdirectory_name="flux_relight_example",
        n_results=num_results,
        torch_precision=torch_precision,
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix=f"FLUX {title} Comparison",
        device=device,
        save_every=50,
        model_name='FLUX',
        pretrained_source='flux',
        seed=global_seed
    )